# Exploration des données — DVF Paris × Transports × Espaces Verts × Équipements

Ce notebook réalise l'ensemble de l'analyse exploratoire du projet :

1. Chargement et nettoyage du dataset DVF
2. EDA sur les prix au m² à Paris
3. Chargement et nettoyage du dataset stations IDF
4. Croisement DVF × Transports (BallTree haversine)
5. Analyse de l'impact des transports sur le prix au m²
6. Chargement et nettoyage du dataset Espaces Verts
7. Croisement DVF × Espaces Verts
8. Analyse de l'impact des espaces verts sur le prix au m²
9. Chargement et nettoyage de la BPE (équipements INSEE)
10. Croisement DVF × Équipements
11. Analyse de l'impact des équipements sur le prix au m²
12. Sauvegarde du dataset enrichi

**Sources de données :**
- `data/raw/dvf.csv` — Demandes de valeurs foncières (open data)
- `data/external/emplacement-des-gares-idf.csv` — Stations IDF (IDFM open data)
- `data/external/espaces_verts.csv` — Espaces verts de Paris (OpenData Paris)
- `data/external/BPE24.csv` — Base Permanente des Équipements 2024 (INSEE)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.neighbors import BallTree

os.makedirs('../plots', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

EARTH_RADIUS_KM = 6371.0
print('Librairies chargées.')

## 1. Chargement et nettoyage du dataset DVF

In [ ]:
dvf_raw = pd.read_csv('../data/raw/dvf.csv', low_memory=False)
print(f'DVF brut : {dvf_raw.shape[0]:,} lignes, {dvf_raw.shape[1]} colonnes')
dvf_raw.head(2)

In [ ]:
df = dvf_raw.copy()

df['valeur_fonciere'] = pd.to_numeric(df['valeur_fonciere'], errors='coerce')
df['surface_reelle_bati'] = pd.to_numeric(df['surface_reelle_bati'], errors='coerce')
df['code_departement'] = df['code_departement'].astype(str).str.strip()
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

n0 = len(df)
df = df[df['nature_mutation'] == 'Vente']
df = df[df['type_local'] == 'Appartement']
df = df[df['code_departement'] == '75']
df = df[df['valeur_fonciere'] > 0]
df = df[df['surface_reelle_bati'] > 0]
df = df[df['latitude'].notna() & df['longitude'].notna()]
n1 = len(df)

df['prix_m2'] = df['valeur_fonciere'] / df['surface_reelle_bati']
df = df[(df['prix_m2'] >= 3000) & (df['prix_m2'] <= 25000)]
df = df.reset_index(drop=True)
n2 = len(df)

print(f'Lignes brutes              : {n0:>10,}')
print(f'Après filtrage             : {n1:>10,}')
print(f'Après suppression outliers : {n2:>10,}')
print(f'Prix m² médian             : {df["prix_m2"].median():>10,.0f} €/m²')
df[['valeur_fonciere', 'surface_reelle_bati', 'prix_m2', 'latitude', 'longitude']].head(3)

## 2. EDA — Prix au m² à Paris

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(df['prix_m2'], bins=80, color='steelblue', edgecolor='white', linewidth=0.4)
ax.axvline(df['prix_m2'].median(), color='tomato', linestyle='--', linewidth=1.5,
           label=f'Médiane : {df["prix_m2"].median():,.0f} €/m²')
ax.axvline(df['prix_m2'].mean(), color='orange', linestyle='--', linewidth=1.5,
           label=f'Moyenne : {df["prix_m2"].mean():,.0f} €/m²')
ax.set_xlabel('Prix au m² (€/m²)')
ax.set_ylabel('Nombre de transactions')
ax.set_title('Distribution des prix au m² — Appartements Paris')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,} €'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()
plt.tight_layout()
plt.savefig('../plots/01_distribution_prix_m2.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df['code_postal'] = df['code_postal'].astype(str).str.strip()
median_by_arr = df.groupby('code_postal')['prix_m2'].median().sort_values(ascending=False)

print('=== Top 5 arrondissements les plus chers ===')
print(median_by_arr.head(5).apply(lambda x: f'{x:,.0f} €/m²').to_string())
print()
print('=== Top 5 arrondissements les moins chers ===')
print(median_by_arr.tail(5).apply(lambda x: f'{x:,.0f} €/m²').to_string())

In [ ]:
arr_sorted = median_by_arr.sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(arr_sorted.index, arr_sorted.values, color='steelblue', edgecolor='white')
ax.set_xlabel('Prix m² médian (€/m²)')
ax.set_title('Prix m² médian par arrondissement — Appartements Paris')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,} €'))
for bar, val in zip(bars, arr_sorted.values):
    ax.text(val + 50, bar.get_y() + bar.get_height() / 2,
            f'{int(val):,} €', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../plots/01_prix_m2_par_arrondissement.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(df[df['surface_reelle_bati'] <= 200]['surface_reelle_bati'], bins=60,
        color='steelblue', edgecolor='white', linewidth=0.4)
ax.axvline(df['surface_reelle_bati'].median(), color='tomato', linestyle='--', linewidth=1.5,
           label=f'Médiane : {df["surface_reelle_bati"].median():.0f} m²')
ax.set_xlabel('Surface réelle bâtie (m²)')
ax.set_ylabel('Nombre de transactions')
ax.set_title('Distribution des surfaces — Appartements Paris (≤ 200 m²)')
ax.legend()
plt.tight_layout()
plt.savefig('../plots/01_distribution_surface.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df['nombre_pieces_principales'] = pd.to_numeric(df['nombre_pieces_principales'], errors='coerce')
median_by_pieces = (
    df[df['nombre_pieces_principales'].between(1, 7)]
    .groupby('nombre_pieces_principales')['prix_m2'].median().sort_index()
)
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(median_by_pieces.index.astype(int), median_by_pieces.values,
              color='steelblue', edgecolor='white')
ax.set_xlabel('Nombre de pièces principales')
ax.set_ylabel('Prix m² médian (€/m²)')
ax.set_title('Prix m² médian par nombre de pièces — Appartements Paris')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,} €'))
for bar, val in zip(bars, median_by_pieces.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            f'{int(val):,} €', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('../plots/01_prix_m2_par_nb_pieces.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Chargement et nettoyage du dataset Stations IDF

Source : Île-de-France Mobilités (IDFM) — open data  
Fichier : `data/external/emplacement-des-gares-idf.csv`

In [ ]:
stations_raw = pd.read_csv('../data/external/emplacement-des-gares-idf.csv', sep=';', low_memory=False)
coords = stations_raw['Geo Point'].str.split(',', expand=True)
stations = stations_raw[['nom_long', 'res_com', 'indice_lig', 'mode', 'exploitant']].copy()
stations['latitude_station'] = pd.to_numeric(coords[0], errors='coerce')
stations['longitude_station'] = pd.to_numeric(coords[1], errors='coerce')
stations = stations.dropna(subset=['latitude_station', 'longitude_station']).reset_index(drop=True)

print(f'Stations nettoyées : {len(stations):,}')
print(f'Modes : {sorted(stations["mode"].dropna().unique())}')

## 4. Croisement DVF × Transports — BallTree (Haversine)

In [ ]:
dvf_coords_rad = np.radians(df[['latitude', 'longitude']].values)
stations_coords_rad = np.radians(stations[['latitude_station', 'longitude_station']].values)
tree_transport = BallTree(stations_coords_rad, metric='haversine')

dist_rad, idx = tree_transport.query(dvf_coords_rad, k=1)
df['distance_station_plus_proche'] = dist_rad[:, 0] * EARTH_RADIUS_KM * 1000
df['nom_station_plus_proche'] = stations.loc[idx[:, 0], 'nom_long'].values
df['mode_station_plus_proche'] = stations.loc[idx[:, 0], 'mode'].values
df['ligne_station_plus_proche'] = stations.loc[idx[:, 0], 'indice_lig'].values

r500 = 500 / (EARTH_RADIUS_KM * 1000)
r1000 = 1000 / (EARTH_RADIUS_KM * 1000)
df['nb_stations_500m'] = tree_transport.query_radius(dvf_coords_rad, r=r500, count_only=True)
df['nb_stations_1000m'] = tree_transport.query_radius(dvf_coords_rad, r=r1000, count_only=True)

print('Transports : OK')
print(f'Distance médiane à la station : {df["distance_station_plus_proche"].median():.0f} m')

## 5. EDA — Impact des transports sur le prix au m²

In [ ]:
bins = [0, 250, 500, 750, 1000, np.inf]
labels_tranches = ['0–250m', '250–500m', '500–750m', '750–1000m', '+1000m']
df['tranche_distance'] = pd.cut(df['distance_station_plus_proche'], bins=bins, labels=labels_tranches)
median_by_tranche = df.groupby('tranche_distance', observed=True)['prix_m2'].median()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels_tranches, median_by_tranche.values, color='steelblue', edgecolor='white')
ax.set_xlabel('Tranche de distance à la station')
ax.set_ylabel('Prix m² médian (€/m²)')
ax.set_title('Prix m² médian par tranche de distance à la station de transport')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,} €'))
for bar, val in zip(bars, median_by_tranche.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            f'{int(val):,} €', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('../plots/02_prix_m2_par_tranche_distance_transport.png', dpi=150, bbox_inches='tight')
plt.show()

corr_t = df['distance_station_plus_proche'].corr(df['prix_m2'])
print(f'Corrélation distance station / prix_m2 : {corr_t:.4f}')

## 6. Chargement et nettoyage du dataset Espaces Verts Paris

Source : OpenData Paris  
Fichier : `data/external/espaces_verts.csv`

In [ ]:
ev_raw = pd.read_csv('../data/external/espaces_verts.csv', sep=';', low_memory=False)
coords_ev = ev_raw['Geo point'].str.split(',', expand=True)
ev = ev_raw[['Nom de l\'espace vert', 'Typologie d\'espace vert', 'Surface calculée']].copy()
ev.columns = ['nom_ev', 'type_ev', 'surface_ev']
ev['latitude_ev'] = pd.to_numeric(coords_ev[0], errors='coerce')
ev['longitude_ev'] = pd.to_numeric(coords_ev[1], errors='coerce')
ev['surface_ev'] = pd.to_numeric(ev['surface_ev'], errors='coerce')
ev = ev.dropna(subset=['latitude_ev', 'longitude_ev']).reset_index(drop=True)

print(f'Espaces verts nettoyés : {len(ev):,}')

## 7. Croisement DVF × Espaces Verts — BallTree (Haversine)

In [ ]:
ev_coords_rad = np.radians(ev[['latitude_ev', 'longitude_ev']].values)
tree_ev = BallTree(ev_coords_rad, metric='haversine')

dist_ev_rad, idx_ev = tree_ev.query(dvf_coords_rad, k=1)
df['distance_ev_plus_proche'] = dist_ev_rad[:, 0] * EARTH_RADIUS_KM * 1000
df['surface_ev_plus_proche'] = ev.loc[idx_ev[:, 0], 'surface_ev'].values
df['nb_ev_500m'] = tree_ev.query_radius(dvf_coords_rad, r=r500, count_only=True)

print('Espaces verts : OK')
print(f'Distance médiane à l\'espace vert : {df["distance_ev_plus_proche"].median():.0f} m')

## 8. EDA — Impact des espaces verts sur le prix au m²

In [ ]:
bins_ev = [0, 100, 200, 300, 500, np.inf]
labels_ev = ['0–100m', '100–200m', '200–300m', '300–500m', '+500m']
df['tranche_distance_ev'] = pd.cut(df['distance_ev_plus_proche'], bins=bins_ev, labels=labels_ev)
median_ev_tranche = df.groupby('tranche_distance_ev', observed=True)['prix_m2'].median()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels_ev, median_ev_tranche.values, color='mediumseagreen', edgecolor='white')
ax.set_xlabel('Tranche de distance à l\'espace vert le plus proche')
ax.set_ylabel('Prix m² médian (€/m²)')
ax.set_title('Prix m² médian par tranche de distance à l\'espace vert le plus proche')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,} €'))
for bar, val in zip(bars, median_ev_tranche.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            f'{int(val):,} €', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('../plots/03_prix_m2_par_distance_espace_vert.png', dpi=150, bbox_inches='tight')
plt.show()

corr_ev = df['distance_ev_plus_proche'].corr(df['prix_m2'])
print(f'Corrélation distance espace vert / prix_m2 : {corr_ev:.4f}')

## 9. Chargement et nettoyage de la BPE — Équipements INSEE 2024

Source : INSEE — Base Permanente des Équipements 2024  
Fichier : `data/external/BPE24.csv`

On conserve uniquement les équipements parisiens susceptibles d'influencer le prix au m² :

| Catégorie | Codes |
|---|---|
| Éducation | A501, A502, A503 |
| Santé | D265, D307, D269 |
| Commerces | B201, B302, B310, B311 |
| Loisirs | F101, F111 |

In [ ]:
CODES_BPE = {
    'education': ['A501', 'A502', 'A503'],
    'sante':     ['D265', 'D307', 'D269'],
    'commerce':  ['B201', 'B302', 'B310', 'B311'],
    'loisirs':   ['F101', 'F111'],
}
ALL_CODES = [c for codes in CODES_BPE.values() for c in codes]

bpe_raw = pd.read_csv(
    '../data/external/BPE24.csv',
    sep=';',
    low_memory=False,
    usecols=['DEP', 'TYPEQU', 'LATITUDE', 'LONGITUDE']
)

bpe = bpe_raw[
    (bpe_raw['DEP'] == '75') &
    (bpe_raw['TYPEQU'].isin(ALL_CODES))
].copy()

bpe['LATITUDE'] = pd.to_numeric(bpe['LATITUDE'], errors='coerce')
bpe['LONGITUDE'] = pd.to_numeric(bpe['LONGITUDE'], errors='coerce')
bpe = bpe.dropna(subset=['LATITUDE', 'LONGITUDE']).reset_index(drop=True)

print(f'Équipements retenus (Paris) : {len(bpe):,}')
print()
for cat, codes in CODES_BPE.items():
    n = bpe['TYPEQU'].isin(codes).sum()
    print(f'  {cat:12s} : {n:,}')

## 10. Croisement DVF × Équipements — BallTree par catégorie

In [ ]:
for cat, codes in CODES_BPE.items():
    subset = bpe[bpe['TYPEQU'].isin(codes)].reset_index(drop=True)
    coords_rad = np.radians(subset[['LATITUDE', 'LONGITUDE']].values)
    tree = BallTree(coords_rad, metric='haversine')

    dist_rad, _ = tree.query(dvf_coords_rad, k=1)
    df[f'distance_{cat}_plus_proche'] = dist_rad[:, 0] * EARTH_RADIUS_KM * 1000
    df[f'nb_{cat}_500m'] = tree.query_radius(dvf_coords_rad, r=r500, count_only=True)

    print(f'{cat:12s} — distance médiane : {df[f"distance_{cat}_plus_proche"].median():.0f} m '
          f'| nb médian 500m : {df[f"nb_{cat}_500m"].median():.0f}')

## 11. EDA — Impact des équipements sur le prix au m²

In [ ]:
# Corrélations de toutes les features de distance avec le prix au m²
distance_cols = [c for c in df.columns if c.startswith('distance_') or c.startswith('nb_')]
correlations = df[distance_cols + ['prix_m2']].corr()['prix_m2'].drop('prix_m2').sort_values()

print('=== Corrélations (Pearson) avec le prix au m² ===')
print(correlations.round(4).to_string())

In [ ]:
# Bar chart des corrélations
fig, ax = plt.subplots(figsize=(11, 6))
colors = ['tomato' if v < 0 else 'steelblue' for v in correlations.values]
ax.barh(correlations.index, correlations.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Corrélation de Pearson avec le prix au m²')
ax.set_title('Corrélation des features de proximité avec le prix au m²')
plt.tight_layout()
plt.savefig('../plots/04_correlations_features_prix_m2.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Prix m² médian par tranche de distance à l'école la plus proche
bins_edu = [0, 200, 400, 600, 800, np.inf]
labels_edu = ['0–200m', '200–400m', '400–600m', '600–800m', '+800m']
df['tranche_distance_education'] = pd.cut(
    df['distance_education_plus_proche'], bins=bins_edu, labels=labels_edu
)
median_edu = df.groupby('tranche_distance_education', observed=True)['prix_m2'].median()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels_edu, median_edu.values, color='steelblue', edgecolor='white')
ax.set_xlabel('Tranche de distance à l\'école la plus proche')
ax.set_ylabel('Prix m² médian (€/m²)')
ax.set_title('Prix m² médian par distance à l\'école la plus proche')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,} €'))
for bar, val in zip(bars, median_edu.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            f'{int(val):,} €', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('../plots/04_prix_m2_par_distance_ecole.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Sauvegarde du dataset enrichi

In [ ]:
output_path = '../data/processed/dvf_paris_enriched.csv'
df.to_csv(output_path, index=False)

print(f'Dataset enrichi sauvegardé : {output_path}')
print(f'Shape finale : {df.shape}')
print()
new_cols = [c for c in df.columns if any(c.startswith(p) for p in
            ['distance_', 'nb_', 'nom_station', 'mode_station', 'ligne_station', 'surface_ev'])]
print(f'Features ajoutées ({len(new_cols)}) :')
for c in new_cols:
    print(f'  - {c}')